# Marketplace quickstart

A working connection to the governed Trino coordinator. Run the cells in order.

Everything you can see follows your **Keycloak group**, not this notebook.


## 1. Connect

Open this notebook in VS Code with the Python and Jupyter extensions. Select your Python environment with `trino`, `pandas`, and `ipykernel` installed.

The first query opens Keycloak in your browser. Sign in as the same username you enter below. No password or access token is stored in the notebook.


In [ ]:
import pathlib, sys

# Make `marketplace` importable no matter where the notebook is opened from.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "marketplace" / "auth.py").exists())
sys.path.insert(0, str(ROOT))

from marketplace.auth import browser_connect, ca_bundle

print("CA certificate:", ca_bundle())


In [ ]:
USER = "alice.nakato"      # your work account

# The first query opens Keycloak. Sign in as USER, including when switching accounts.
conn = browser_connect(USER, catalog="iceberg", schema="marketplace")

cur = conn.cursor()
cur.execute("SELECT current_user, current_groups()")
principal, groups = cur.fetchone()
print(f"connected as {principal}, groups: {list(groups)}")


## 2. What can I see?

If this list is empty, the account is in no group. That is an identity problem,
not a permissions one — see the troubleshooting cell at the bottom.

In [ ]:
cur.execute("SHOW TABLES FROM marketplace")
for (table,) in cur.fetchall():
    print(" ", table)


## 3. Query a certified product

In [ ]:
import pandas as pd

# pandas warns about non-SQLAlchemy connections, so take the rows from the
# cursor directly - the column names come back on cursor.description.
cur.execute("""
    SELECT process, workflow_stage, avg_waiting_days, pct_time_waiting
    FROM marketplace.delay_analysis
    WHERE process = 'GMP'
    ORDER BY avg_waiting_days DESC
    LIMIT 10
""")
pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])


## 4. Your sandbox

A view stores a definition, not data — about 600 bytes, recomputed on read and
never stale. Prefer it to a table unless you specifically need a frozen copy.

In [ ]:
from marketplace.products import sandbox_schema

schema = sandbox_schema(USER)
cur.execute(f"""
    CREATE OR REPLACE VIEW {schema}.my_slow_stages AS
    SELECT process, workflow_stage, avg_elapsed_days
    FROM marketplace.delay_analysis
    WHERE avg_elapsed_days > 20
""")
cur.fetchall()
print("created", f"{schema}.my_slow_stages")


## 5. When something is denied

Run this rather than guessing. It separates *who Trino thinks you are* from
*what that is allowed to do*, which is almost always where the confusion is.

In [ ]:
# Inspect the SAME browser-authenticated connection; do not request a password token.
cur.execute("SELECT current_user, current_groups()")
print("Identity and roles:", cur.fetchone())
cur.execute("SHOW SCHEMAS FROM iceberg")
print("Visible schemas:", [row[0] for row in cur.fetchall()])
# If groups are empty, the platform operator should verify Keycloak membership
# and run: python -m marketplace.identity sync
